# Plan a budget for continuous evaluations

Compare three judges on the same support answer. Then create a sampled continuous evaluation and a pass-rate monitor in a **new, empty Observe project**. The example pauses its task and leaves its monitor muted; it does not send alerts or score anyone else's traffic. You need Future AGI API credentials with permission to create projects, evaluations and monitors.

## Install and authenticate

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7", "requests"], check=True);

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    value = getpass(f"{name}: ")
    if not value.strip():
        raise ValueError(f"{name} is required")
    os.environ[name] = value.strip()

for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    load_secret(name)

## Compare candidate judges

Give each model the same question and answer. One example is a smoke check: compare the verdicts against human labels over representative traffic before choosing a production judge.

In [ ]:
from fi.evals import Evaluator, evaluate

sample = {
    "input": "How can I return my unused jacket?",
    "output": "Bring the unused jacket and receipt to a Ridgeline store within 30 days of delivery.",
}
history = Evaluator()
for model in ("turing_flash", "turing_small", "turing_large"):
    verdict = evaluate("is_helpful", model=model, **sample)
    if verdict.passed is None or verdict.error:
        raise RuntimeError(f"No {model} verdict: {verdict.error or verdict.reason}")
    eval_id = verdict.metadata["eval_id"]
    record = history.get_eval_result(eval_id)["result"]["result"]
    assert record["eval_id"] == eval_id and record["name"] == "is_helpful"
    print(model, "|", record["value"], "|", verdict.reason)

## Estimate the sample size

A 10% sample of 50,000 eligible spans gives around 5,000 judge calls per day with one evaluator. It is not a hard cap on calls, tokens or credits: traffic volume, the number of evaluators, and input lengths all matter. Smaller samples can also miss rare failures.

In [ ]:
eligible_spans_per_day = 50_000
sampling_rate = 10
evaluators_per_span = 1
estimated_calls = eligible_spans_per_day * sampling_rate / 100 * evaluators_per_span
print(f"Estimated judge calls per day: {estimated_calls:,.0f}")

## Set up a project and evaluator

The notebook uses a fresh project each run to avoid changing active evaluations. The mapping below assumes spans carry `input.value` and `output.value`; when adapting it, use the attribute paths your instrumentation actually sends.

In [ ]:
import requests
from uuid import uuid4

base = os.getenv("FI_BASE_URL", "https://api.futureagi.com").rstrip("/")
session = requests.Session()
session.headers.update({"X-Api-Key": os.environ["FI_API_KEY"], "X-Secret-Key": os.environ["FI_SECRET_KEY"]})

def api(method, path, **kwargs):
    response = session.request(method, base + path, timeout=45, **kwargs)
    response.raise_for_status()
    data = response.json()
    return data.get("result", data)

name = f"ridgeline-eval-budget-{uuid4().hex[:10]}"
project_id = api("POST", "/tracer/project/", json={
    "name": name, "model_type": "GenerativeLLM", "trace_type": "observe",
})["project_id"]
assert api("GET", f"/tracer/project/{project_id}/")["name"] == name
catalog = api("GET", "/sdk/api/v1/get-evals/")
template = next(t for t in catalog if t["name"] == "is_helpful")
config_id = api("POST", "/tracer/custom-eval-config/", json={
    "project": project_id, "name": "Return answer helpfulness", "eval_template": template["id"],
    "model": "turing_flash", "mapping": {"input": "input.value", "output": "output.value"},
})["id"]
configs = api("GET", "/tracer/custom-eval-config/list_custom_eval_configs/", params={"project_id": project_id, "fresh": "true"})
assert any(c["id"] == config_id and c["model"] == "turing_flash" for c in configs)
print("Observe project:", name, "| evaluator:", config_id)

## Create a continuous task at the planned rate

This task is configured for new spans in the demo project. A continuous task has no fixed run size. To run a fixed-size backfill instead, use a historical task.

In [ ]:
task_id = api("POST", "/tracer/eval-task/", json={
    "project": project_id, "name": "Sample support answers", "evals": [config_id],
    "run_type": "continuous", "row_type": "spans", "sampling_rate": sampling_rate,
})["id"]
task_path = f"/tracer/eval-task/{task_id}/"
task = api("GET", task_path)
assert task["project"] == project_id and task["run_type"] == "continuous"
assert task["sampling_rate"] == sampling_rate and config_id in task["evals"]
print("Sample rate:", task["sampling_rate"], "% | task:", task_id)

## Monitor the fraction of passing answers

Configure an hourly warning below a 90% pass rate and a critical threshold below 80%. Leave delivery disabled in the demo. In production, inspect your baseline, choose thresholds and a notification destination, then enable the monitor.

In [ ]:
monitor_id = api("POST", "/tracer/alerts/", json={
    "kind": "metric", "project": project_id, "name": "Sampled helpfulness pass rate",
    "metric_type": "evaluation_metrics", "metric": config_id, "threshold_metric_value": "Passed",
    "threshold_type": "static", "threshold_operator": "less_than",
    "critical_threshold_value": 0.80, "warning_threshold_value": 0.90,
    "alert_frequency": 60, "enabled": False, "notification_emails": [],
})["id"]
monitor = api("GET", f"/tracer/alerts/metric/{monitor_id}/")
assert monitor["metric"] == config_id and monitor["enabled"] is False
assert monitor["warning_threshold_value"] == 0.90 and monitor["critical_threshold_value"] == 0.80
print("Muted monitor:", monitor_id, "| metric:", monitor["metric_name"])

## Pause the demo task

Pause only the task created above. You can inspect its sampling settings and the muted monitor in Observe without leaving a background evaluator running.

In [ ]:
import time

for _ in range(30):
    state = api("GET", task_path)["status"]
    if state != "pending":
        break
    time.sleep(2)
if state == "running":
    api("POST", "/tracer/eval-task/pause_eval_task/", params={"eval_task_id": task_id})
state = api("GET", task_path)["status"]
if state != "paused":
    raise RuntimeError(f"Task {task_id} is {state}, not paused. Check its status in Observe.")
print("Task paused; monitor muted.")